# Module 3, Lab 1: Multimodal GenAI - Image Analysis for IT Infrastructure

## Objective
In this lab, learners will use a multimodal Generative AI model to analyze IT infrastructure images such as server racks, network topology diagrams, dashboard screenshots, error screenshots, and monitoring visualizations. Participants will submit images to the model, extract insights, identify potential issues, generate summaries, and create incident reports to support IT operations and troubleshooting workflows.

## Learning Outcomes
- Load and display IT infrastructure images
- Use multimodal AI models (OpenAI Vision API) to analyze images
- Extract structured insights from infrastructure images
- Identify anomalies and potential issues
- Generate incident reports and recommendations

## Prerequisites
- OpenAI API key with vision model access
- Python 3.8+
- Required libraries: openai, pillow, python-dotenv, matplotlib, requests

## Setup and Imports

In [ ]:
import os
import base64
import json
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
from PIL import Image
import requests
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# Create directories for outputs
Path('sample_images').mkdir(exist_ok=True)
Path('analysis_results').mkdir(exist_ok=True)
Path('incident_reports').mkdir(exist_ok=True)

print("✓ Environment initialized")
print(f"✓ OpenAI API Key loaded: {bool(os.getenv('OPENAI_API_KEY'))}")

## Step 1: Load and Display Infrastructure Images

In [ ]:
def display_image(image_path, title=""):
    """
    Display an image with title
    
    Args:
        image_path: Path to the image file
        title: Title to display above the image
    """
    try:
        img = Image.open(image_path)
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title(title, fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        print(f"✓ Image loaded: {image_path}")
        print(f"  Size: {img.size[0]} x {img.size[1]} pixels")
        return img
    except FileNotFoundError:
        print(f"✗ Image not found: {image_path}")
        return None

def list_images(directory='sample_images'):
    """
    List all images in a directory
    """
    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.gif']:
        image_files.extend(Path(directory).glob(ext))
    
    if not image_files:
        print(f"No images found in {directory}")
        print("Please run the image generator script first: python generate_sample_images.py")
        return []
    
    print(f"✓ Found {len(image_files)} images:")
    for i, img_path in enumerate(image_files, 1):
        print(f"  {i}. {img_path.name}")
    
    return sorted(image_files)

# List available images
images = list_images()

## Step 2: Image Encoding Function for API

In [ ]:
def encode_image_to_base64(image_path):
    """
    Encode image to base64 for API submission
    
    Args:
        image_path: Path to the image file
    
    Returns:
        base64_string: Encoded image as base64
    """
    with open(image_path, 'rb') as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def get_image_media_type(image_path):
    """
    Determine media type based on file extension
    """
    ext = Path(image_path).suffix.lower()
    media_types = {
        '.jpg': 'image/jpeg',
        '.jpeg': 'image/jpeg',
        '.png': 'image/png',
        '.gif': 'image/gif',
        '.webp': 'image/webp'
    }
    return media_types.get(ext, 'image/jpeg')

print("✓ Image encoding functions ready")

## Step 3: Analyze Images with OpenAI Vision API

In [ ]:
def analyze_infrastructure_image(image_path, analysis_type='general'):
    """
    Analyze IT infrastructure image using OpenAI Vision API
    
    Args:
        image_path: Path to the image file
        analysis_type: Type of analysis ('general', 'server_rack', 'dashboard', 'network', 'error')
    
    Returns:
        analysis_result: Dictionary with analysis results
    """
    
    # Define analysis prompts for different infrastructure types
    analysis_prompts = {
        'general': """Analyze this IT infrastructure image and provide:
1. Description of what you see
2. Key components identified
3. Potential issues or concerns
4. Health status assessment
5. Recommendations""",
        
        'server_rack': """Analyze this server rack image and provide:
1. Number and type of servers/equipment visible
2. Cable management assessment
3. Power distribution status
4. Cooling/ventilation observations
5. Potential issues (e.g., overcrowding, cable mess, heat hotspots)
6. Recommendations for optimization""",
        
        'dashboard': """Analyze this monitoring dashboard screenshot and provide:
1. Key metrics visible
2. Current status of each metric (normal, warning, critical)
3. Trends observed
4. Any anomalies or red flags
5. Alert status and severity
6. Recommended actions""",
        
        'network': """Analyze this network topology diagram and provide:
1. Network components identified
2. Network architecture description
3. Connectivity assessment
4. Redundancy evaluation
5. Potential bottlenecks or failure points
6. Security considerations
7. Recommendations for improvement""",
        
        'error': """Analyze this error screenshot and provide:
1. Error message and type
2. Affected component/service
3. Severity assessment
4. Likely root causes
5. Immediate actions required
6. Long-term remediation steps"""
    }
    
    prompt = analysis_prompts.get(analysis_type, analysis_prompts['general'])
    
    try:
        # Encode image
        image_base64 = encode_image_to_base64(image_path)
        media_type = get_image_media_type(image_path)
        
        # Call OpenAI Vision API
        response = client.chat.completions.create(
            model="gpt-4o",
            max_tokens=1024,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": prompt
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:{media_type};base64,{image_base64}"
                            }
                        }
                    ]
                }
            ]
        )
        
        analysis_text = response.choices[0].message.content
        
        result = {
            'image_path': str(image_path),
            'analysis_type': analysis_type,
            'timestamp': datetime.now().isoformat(),
            'analysis': analysis_text,
            'model': 'gpt-4o',
            'tokens_used': response.usage.total_tokens
        }
        
        return result
    
    except Exception as e:
        print(f"✗ Error analyzing image: {str(e)}")
        return None

print("✓ Image analysis function ready")

## Step 4: Extract Structured Insights

In [ ]:
def extract_structured_insights(image_path):
    """
    Extract structured insights from infrastructure image
    Returns JSON-structured analysis
    """
    
    prompt = """Analyze this IT infrastructure image and return ONLY a JSON object (no markdown, no extra text) with this exact structure:
{
  "image_type": "server_rack|dashboard|network|error|other",
  "health_status": "healthy|warning|critical",
  "components": ["component1", "component2"],
  "issues": ["issue1", "issue2"],
  "metrics": {
    "cpu_usage": "value if visible",
    "memory_usage": "value if visible",
    "disk_usage": "value if visible",
    "network_traffic": "value if visible"
  },
  "severity": "low|medium|high|critical",
  "recommendations": ["rec1", "rec2"]
}"""
    
    try:
        image_base64 = encode_image_to_base64(image_path)
        media_type = get_image_media_type(image_path)
        
        response = client.chat.completions.create(
            model="gpt-4o",
            max_tokens=1024,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": prompt
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:{media_type};base64,{image_base64}"
                            }
                        }
                    ]
                }
            ]
        )
        
        response_text = response.choices[0].message.content
        
        # Strip markdown code fences if present
        cleaned_text = response_text.strip()
        if cleaned_text.startswith('```'):
            cleaned_text = cleaned_text.split('```')[1]
            if cleaned_text.startswith('json'):
                cleaned_text = cleaned_text[4:]
            cleaned_text = cleaned_text.strip()
        
        # Parse JSON from response
        try:
            insights = json.loads(cleaned_text)
            return insights
        except json.JSONDecodeError:
            print(f"Warning: Could not parse JSON response. Raw response:\n{response_text}")
            return {"raw_response": response_text}
    
    except Exception as e:
        print(f"✗ Error extracting insights: {str(e)}")
        return None

print("✓ Structured insights extraction ready")

## Step 5: Generate Incident Reports

In [ ]:
def generate_incident_report(image_path, analysis_result):
    """
    Generate formal incident report from image analysis
    
    Args:
        image_path: Path to the image
        analysis_result: Result from analyze_infrastructure_image()
    
    Returns:
        incident_report: Formatted incident report
    """
    
    if not analysis_result:
        return None
    
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    image_name = Path(image_path).name
    
    # Generate ticket ID
    ticket_id = f"INC-{datetime.now().strftime('%Y%m%d%H%M%S')}"
    
    report = f"""================================================================================
                         IT INFRASTRUCTURE INCIDENT REPORT
================================================================================

TICKET ID:           {ticket_id}
REPORT DATE:         {timestamp}
SOURCE IMAGE:        {image_name}
ANALYSIS MODEL:      {analysis_result.get('model', 'Unknown')}

--------------------------------------------------------------------------------
AUTOMATED ANALYSIS SUMMARY
--------------------------------------------------------------------------------

{analysis_result.get('analysis', 'No analysis available')}

--------------------------------------------------------------------------------
NEXT STEPS
--------------------------------------------------------------------------------

1. Review the automated analysis above
2. Escalate to appropriate team if severity is HIGH or CRITICAL
3. Follow incident management procedures for response
4. Document all actions taken
5. Update ticket status and add notes

--------------------------------------------------------------------------------
METADATA
--------------------------------------------------------------------------------

Model Used:          {analysis_result.get('model', 'Unknown')}
Analysis Type:       {analysis_result.get('analysis_type', 'general')}
Tokens Used:         {analysis_result.get('tokens_used', 'N/A')}
Generated At:        {analysis_result.get('timestamp', 'N/A')}

================================================================================
"""
    
    return report

def save_incident_report(report, ticket_id=None):
    """
    Save incident report to file
    """
    if not ticket_id:
        ticket_id = f"INC-{datetime.now().strftime('%Y%m%d%H%M%S')}"
    
    filename = f"incident_reports/{ticket_id}.txt"
    
    with open(filename, 'w') as f:
        f.write(report)
    
    print(f"✓ Report saved: {filename}")
    return filename

print("✓ Incident report functions ready")

## Step 6: Complete Analysis Workflow

In [ ]:
def complete_image_analysis_workflow(image_path, analysis_type='general'):
    """
    Execute complete analysis workflow for an image
    
    Args:
        image_path: Path to the image
        analysis_type: Type of analysis to perform
    
    Returns:
        workflow_result: Dictionary with all results
    """
    
    print(f"\n{'='*80}")
    print(f"ANALYZING: {Path(image_path).name}")
    print(f"{'='*80}\n")
    
    # Step 1: Display image
    print("[1/4] Loading and displaying image...")
    img = display_image(image_path, title=f"Infrastructure Image: {Path(image_path).name}")
    
    if img is None:
        return None
    
    # Step 2: Perform analysis
    print("\n[2/4] Analyzing image with vision model...")
    analysis = analyze_infrastructure_image(image_path, analysis_type)
    
    if analysis is None:
        print("✗ Analysis failed")
        return None
    
    print("\n[2b/4] Extracting structured insights...")
    insights = extract_structured_insights(image_path)
    
    # Step 3: Generate report
    print("\n[3/4] Generating incident report...")
    report = generate_incident_report(image_path, analysis)
    report_filename = save_incident_report(report)
    
    # Step 4: Display results
    print("\n[4/4] Results summary:")
    print(f"\n{report}")
    
    # Save structured insights
    insights_filename = f"analysis_results/{Path(image_path).stem}_insights.json"
    with open(insights_filename, 'w') as f:
        json.dump(insights, f, indent=2)
    print(f"✓ Structured insights saved: {insights_filename}")
    
    return {
        'image_path': str(image_path),
        'analysis': analysis,
        'insights': insights,
        'report': report,
        'report_file': report_filename,
        'insights_file': insights_filename
    }

print("✓ Complete workflow function ready")

## Step 7: Batch Analysis

In [ ]:
def batch_analyze_images(image_directory='sample_images', analysis_type='general'):
    """
    Analyze multiple images in batch
    
    Args:
        image_directory: Directory containing images
        analysis_type: Type of analysis
    
    Returns:
        results: List of analysis results
    """
    
    images = list_images(image_directory)
    
    if not images:
        print("No images to analyze")
        return []
    
    results = []
    
    for i, image_path in enumerate(images, 1):
        print(f"\n[{i}/{len(images)}] Processing: {image_path.name}")
        
        try:
            analysis = analyze_infrastructure_image(image_path, analysis_type)
            insights = extract_structured_insights(image_path)
            report = generate_incident_report(image_path, analysis)
            report_filename = save_incident_report(report)
            
            result = {
                'image': str(image_path),
                'status': 'success',
                'report_file': report_filename
            }
            results.append(result)
            
        except Exception as e:
            print(f"✗ Error processing {image_path.name}: {str(e)}")
            results.append({
                'image': str(image_path),
                'status': 'failed',
                'error': str(e)
            })
    
    # Save batch summary
    summary_file = f"analysis_results/batch_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(summary_file, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\n✓ Batch analysis complete. Summary saved: {summary_file}")
    return results

print("✓ Batch analysis function ready")

## EXECUTION EXAMPLES

### Example 1: Analyze a Single Image

In [ ]:
# Example: Analyze server rack image
# Uncomment and run after generating sample images

if images:
    result = complete_image_analysis_workflow(
        image_path=str(images[0]),
        analysis_type='server_rack'
    )
    print("\n✓ Analysis complete!")
else:
    print("No images available. Please run: python generate_sample_images.py")

### Example 2: Batch Analyze All Images

In [ ]:
# Example: Batch analyze all images
# Uncomment and run to process all images in sample_images directory

batch_results = batch_analyze_images(
    image_directory='sample_images',
    analysis_type='general'
)

print(f"\n{'='*80}")
print("BATCH ANALYSIS SUMMARY")
print(f"{'='*80}")
for result in batch_results:
    status = "✓" if result['status'] == 'success' else "✗"
    print(f"{status} {result['image']}")
    if result['status'] == 'success':
        print(f"   Report: {result['report_file']}")

### Example 3: Analyze Specific Infrastructure Type

In [ ]:
# Example: Analyze dashboard screenshot with dashboard-specific prompts
# Uncomment and run after generating sample images

dashboard_image = 'sample_images/02_monitoring_dashboard.png'
if Path(dashboard_image).exists():
    result = complete_image_analysis_workflow(
        image_path=dashboard_image,
        analysis_type='dashboard'
    )
else:
    print(f"Image not found: {dashboard_image}")

## Step 8: Utility Functions for Export

In [ ]:
def export_analysis_to_csv(results_directory='analysis_results'):
    """
    Export all analysis results to CSV for reporting
    """
    import csv
    
    csv_file = f"{results_directory}/analysis_export_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    
    with open(csv_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Image', 'Image Type', 'Health Status', 'Severity', 'Issues Count', 'Timestamp'])
    
    print(f"✓ CSV export template created: {csv_file}")
    return csv_file

def print_summary_stats():
    """
    Print summary statistics of analysis
    """
    report_files = list(Path('incident_reports').glob('*.txt'))
    insight_files = list(Path('analysis_results').glob('*_insights.json'))
    
    print(f"\n{'='*80}")
    print("ANALYSIS SUMMARY")
    print(f"{'='*80}")
    print(f"Total Reports Generated:    {len(report_files)}")
    print(f"Total Insights Extracted:   {len(insight_files)}")
    print(f"\nReport Directory:           {Path('incident_reports').absolute()}")
    print(f"Analysis Directory:         {Path('analysis_results').absolute()}")
    print(f"{'='*80}")

print("✓ Export and utility functions ready")

## Summary

This lab demonstrates:

1. **Image Loading**: Display and verify infrastructure images
2. **API Integration**: Send images to multimodal AI models
3. **Analysis Extraction**: Parse and structure AI responses
4. **Incident Generation**: Create actionable incident reports
5. **Batch Processing**: Analyze multiple images efficiently
6. **Export & Reporting**: Save results for IT operations teams

### Key Skills Learned:
- Working with image files in Python
- Multimodal AI model integration
- Prompt engineering for specific domains (IT operations)
- JSON parsing and structured output
- Batch processing and automation
- Report generation and documentation

### Real-World Applications:
- Automated infrastructure monitoring
- Incident ticket generation
- Visual anomaly detection
- Operational efficiency improvement
- Hands-free troubleshooting assistance